# 04 — Modern Components: RoPE, RMSNorm, SwiGLU, GQA

This notebook covers the **2025-era** components used in Llama, Gemma, Mistral, and SmolLM.
Most tutorials still teach GPT-2's architecture (2019). These are the modern replacements:

| Old (GPT-2, 2019) | Modern (Llama/Gemma, 2024-2025) |
|---|---|
| Learned positional embeddings | **RoPE** (Rotary Position Embeddings) |
| LayerNorm | **RMSNorm** |
| GELU activation | **SwiGLU** |
| Multi-Head Attention | **Grouped Query Attention (GQA)** |

We implement each from scratch in numpy to understand the math.

---

## Key Terminology

| Term | Plain English | Origin / Context |
|---|---|---|
| **RoPE** (Rotary Position Embeddings) | Encode position by rotating Q/K vectors in 2D planes. The angle of rotation = position × frequency. | Su et al. (2021), "RoFormer". Used by Llama, Gemma, Mistral, Qwen — essentially all 2024+ models. Replaced learned absolute positional embeddings. |
| **Relative position** | The distance between two tokens (e.g., token 5 and token 3 are 2 apart). Contrast with *absolute* position (token 5 is at index 5). | RoPE naturally encodes relative position because when computing Q·K, the rotation angles subtract: the score depends on `pos_i - pos_j`. |
| **Frequency / θ** | In RoPE, each dimension pair rotates at a different speed: `θ_i = 10000^(-2i/d)`. Low-index pairs rotate fast (fine position), high-index pairs rotate slow (coarse position). | Analogous to a clock: seconds hand (fast/fine), minutes hand (medium), hours hand (slow/coarse). Together they uniquely identify any time. |
| **Context length extrapolation** | The ability to handle longer sequences at inference than seen during training. | Learned embeddings fail here (no embedding for position 1025 if trained on 1024). RoPE handles it because rotation is continuous — position 1025 is just a slightly further rotation. |
| **RMSNorm** (Root Mean Square Normalization) | Normalize by dividing by the RMS (root mean square) of the vector, then scale by a learned parameter γ. | Zhang & Sennrich (2019). Simpler than LayerNorm (no mean subtraction, no β bias). 7-64% faster. Used by all modern LLMs. |
| **Pre-norm vs Post-norm** | **Pre-norm**: normalize BEFORE the sublayer (attention/FFN). **Post-norm**: normalize AFTER. | GPT-2 used post-norm. All modern models use pre-norm — it's more stable for training deep networks because gradients flow more cleanly through residual connections. |
| **Residual connection** | `output = x + sublayer(x)` — add the input back to the output. Also called "skip connection". | He et al. (2015), ResNet paper. Without residuals, gradients vanish in deep networks. With residuals, gradients have a direct path backward. This is why transformers can have 100+ layers. |
| **SwiGLU** (Swish-Gated Linear Unit) | FFN with gating: `SwiGLU(x) = Swish(x·W_gate) ⊙ (x·W_up)`. The gate controls information flow. | Shazeer (2020), "GLU Variants Improve Transformer". Adopted by PaLM, Llama, Gemma. 1-2% better than GELU at same parameter count. |
| **Swish** activation | `swish(x) = x · sigmoid(x)`. Smooth, non-monotonic (allows small negative values through). | Ramachandran et al. (2017). Also called SiLU (Sigmoid Linear Unit). Replaced GELU in modern architectures. |
| **Gating mechanism** | Element-wise multiplication of two pathways: one controls "what passes through" (gate), the other provides the content. | Inspired by LSTM gates. The gate learns to selectively filter information, acting as a learned "relevance mask". |
| **GQA** (Grouped Query Attention) | Share K/V heads across multiple Q heads. E.g., 6 Q heads share 2 KV heads (3:1 ratio). | Ainslie et al. (2023), "GQA: Training Generalized Multi-Query Transformer Models". Used by Llama 2/3, Gemma, Mistral. |
| **MQA** (Multi-Query Attention) | Extreme case: ALL Q heads share ONE KV head. Maximum memory savings, slight quality loss. | Shazeer (2019). GQA is the middle ground between MHA (no sharing) and MQA (full sharing). |
| **KV-cache** | During text generation, cache the K and V tensors from previous tokens so they don't need recomputation. | Essential for fast inference. Without it, generating token N requires recomputing attention for all N-1 previous tokens. GQA reduces cache size by `n_heads/n_kv_heads` factor. |
| **Weight tying** | Share the same weight matrix for token embeddings and the final output projection (lm_head). | Press & Wolf (2017). Reduces parameter count and improves performance — the embedding and prediction tasks are mathematically related. |
| **FFN** (Feed-Forward Network) | The MLP (multi-layer perceptron) after attention in each transformer block. Projects up (expand), applies activation, projects down (compress). | Each token passes through this independently. Attention moves information between tokens; FFN processes information within each token. |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

---
## Part 1: RoPE (Rotary Position Embeddings)

### Problem with Learned Positional Embeddings
GPT-2 adds a learned vector per position: `embedding[pos] + position_embedding[pos]`
- Fixed max length (can't extrapolate beyond training)
- Position info is added once and gets diluted through layers
- Doesn't naturally encode relative position

### RoPE's Elegant Solution
Instead of ADDING position to the embedding, RoPE **ROTATES** the Q and K vectors
based on their position. The key insight:

> When you take the dot product Q·K (attention score), the rotation angles
> subtract, so the score naturally depends on **relative position** (i - j).

### The Math
For each pair of dimensions (2i, 2i+1), apply a 2D rotation by angle `θ_i * position`:

```
θ_i = 10000^(-2i/d)      — frequency for dimension pair i
angle = position * θ_i    — rotation angle

[x_2i  ]     [cos(angle)  -sin(angle)] [x_2i  ]
[x_2i+1]  =  [sin(angle)   cos(angle)] [x_2i+1]
```

In [ ]:
def precompute_rope_frequencies(dim, max_seq_len, base=10000.0):
    """Precompute the rotation frequencies for RoPE.
    
    Each pair of dimensions gets a different frequency:
    - Low-index pairs: high frequency (fast rotation) → encode fine-grained position
    - High-index pairs: low frequency (slow rotation) → encode coarse position
    
    Returns:
        cos, sin: (max_seq_len, dim) — precomputed rotation components
    """
    # Frequencies for each dimension pair
    # θ_i = 1 / (base^(2i/dim)) for i = 0, 1, ..., dim/2 - 1
    freqs = 1.0 / (base ** (np.arange(0, dim, 2).astype(float) / dim))
    
    # Positions: 0, 1, 2, ..., max_seq_len-1
    positions = np.arange(max_seq_len).astype(float)
    
    # Outer product: angles[pos, i] = pos * θ_i
    angles = np.outer(positions, freqs)  # (max_seq_len, dim/2)
    
    # Duplicate for pairs: [cos(θ), cos(θ), cos(2θ), cos(2θ), ...]
    cos = np.cos(angles)
    sin = np.sin(angles)
    
    return cos, sin


def apply_rope(x, cos, sin):
    """Apply rotary embeddings to Q or K vectors.
    
    x: (seq_len, dim) — query or key vectors
    cos, sin: (seq_len, dim/2) — precomputed rotation
    
    For each pair (x_2i, x_2i+1), rotate by angle:
        x_2i'   = x_2i * cos - x_2i+1 * sin
        x_2i+1' = x_2i * sin + x_2i+1 * cos
    """
    seq_len, dim = x.shape
    
    # Split into even and odd indices
    x_even = x[:, 0::2]   # (seq_len, dim/2)
    x_odd = x[:, 1::2]    # (seq_len, dim/2)
    
    # Apply rotation
    x_even_rot = x_even * cos - x_odd * sin
    x_odd_rot = x_even * sin + x_odd * cos
    
    # Interleave back
    result = np.empty_like(x)
    result[:, 0::2] = x_even_rot
    result[:, 1::2] = x_odd_rot
    
    return result


# Demo
dim = 8
max_seq_len = 16
cos, sin = precompute_rope_frequencies(dim, max_seq_len)

print(f"Frequencies shape: cos={cos.shape}, sin={sin.shape}")
print(f"\nFrequency values (how fast each dimension pair rotates):")
freqs = 1.0 / (10000.0 ** (np.arange(0, dim, 2).astype(float) / dim))
for i, f in enumerate(freqs):
    print(f"  Dim pair {i}: θ = {f:.6f} (period = {2*np.pi/f:.1f} positions)")

In [ ]:
# KEY PROPERTY: RoPE makes attention scores depend on RELATIVE position
# Demo: Q at position i, K at position j → score depends on (i-j)

dim = 8
cos, sin = precompute_rope_frequencies(dim, 20)

# Same vector at two different positions
v = np.random.randn(1, dim)

# Place the same Q at position 0 and same K at various positions
q_base = np.random.randn(1, dim)
k_base = np.random.randn(1, dim)

scores_by_relative_pos = []
for j in range(15):
    # Q at position 5, K at position j
    q_pos = 5
    q_rotated = apply_rope(q_base, cos[q_pos:q_pos+1], sin[q_pos:q_pos+1])
    k_rotated = apply_rope(k_base, cos[j:j+1], sin[j:j+1])
    score = (q_rotated @ k_rotated.T / np.sqrt(dim))[0, 0]
    scores_by_relative_pos.append((j - q_pos, score))

# Now do the same with Q at position 10
scores_shifted = []
for j in range(15):
    q_pos = 10
    q_rotated = apply_rope(q_base, cos[q_pos:q_pos+1], sin[q_pos:q_pos+1])
    k_rotated = apply_rope(k_base, cos[j:j+1], sin[j:j+1])
    score = (q_rotated @ k_rotated.T / np.sqrt(dim))[0, 0]
    scores_shifted.append((j - q_pos, score))

fig, ax = plt.subplots(figsize=(10, 4))
rel_pos_1, scores_1 = zip(*scores_by_relative_pos)
rel_pos_2, scores_2 = zip(*scores_shifted)
ax.plot(rel_pos_1, scores_1, 'o-', color='#2196F3', label='Q at pos 5', linewidth=2)
ax.plot(rel_pos_2, scores_2, 's--', color='#FF5722', label='Q at pos 10', linewidth=2)
ax.set_xlabel('Relative Position (K_pos - Q_pos)', fontsize=12)
ax.set_ylabel('Attention Score', fontsize=12)
ax.set_title('RoPE: Attention Score Depends on Relative Position\n'
             '(same pattern regardless of absolute position)', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("The curves have a similar pattern — RoPE encodes RELATIVE position.")
print("This is why RoPE models can generalize to longer sequences than they were trained on.")

In [ ]:
# Visualize the rotation patterns
dim = 64
max_pos = 50
cos, sin = precompute_rope_frequencies(dim, max_pos)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

im0 = axes[0].imshow(cos, aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
axes[0].set_title('cos(position * θ_i)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Dimension pair index', fontsize=11)
axes[0].set_ylabel('Position', fontsize=11)
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(sin, aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
axes[1].set_title('sin(position * θ_i)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Dimension pair index', fontsize=11)
axes[1].set_ylabel('Position', fontsize=11)
plt.colorbar(im1, ax=axes[1])

plt.suptitle('RoPE Rotation Patterns\n(low dims = fast rotation, high dims = slow rotation)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Left columns rotate quickly (high frequency) — encode fine position differences")
print("Right columns rotate slowly (low frequency) — encode coarse position")
print("This is analogous to how a clock has fast seconds, medium minutes, slow hours.")

---
## Part 2: RMSNorm (Root Mean Square Normalization)

### Why Normalize?
Without normalization, activations grow or shrink as they pass through layers.
This causes training instability — gradients either vanish or explode.

### LayerNorm (Old) vs RMSNorm (Modern)

**LayerNorm**: `(x - mean) / std * γ + β`  
**RMSNorm**: `x / RMS(x) * γ`

RMSNorm is simpler:
- No mean subtraction (the re-centering doesn't help much)
- No bias β
- 7-64% faster than LayerNorm with equivalent performance
- Used by Llama, Gemma, Mistral, SmolLM, and all modern models

In [ ]:
def layer_norm(x, gamma, beta, eps=1e-5):
    """Old-style LayerNorm: normalize by mean and std, then scale and shift."""
    mean = np.mean(x, axis=-1, keepdims=True)
    var = np.var(x, axis=-1, keepdims=True)
    x_norm = (x - mean) / np.sqrt(var + eps)
    return gamma * x_norm + beta


def rms_norm(x, gamma, eps=1e-5):
    """Modern RMSNorm: normalize by RMS only, then scale.
    
    RMS(x) = sqrt(mean(x^2))
    output = x / RMS(x) * gamma
    
    Simpler, faster, no mean subtraction, no beta.
    """
    rms = np.sqrt(np.mean(x ** 2, axis=-1, keepdims=True) + eps)
    return (x / rms) * gamma


# Compare
dim = 8
x = np.random.randn(4, dim) * 5 + 3  # shifted and scaled input
gamma = np.ones(dim)
beta = np.zeros(dim)

ln_out = layer_norm(x, gamma, beta)
rms_out = rms_norm(x, gamma)

print("Input statistics:")
print(f"  Mean: {x.mean(axis=-1).round(3)}")
print(f"  Std:  {x.std(axis=-1).round(3)}")
print(f"  RMS:  {np.sqrt(np.mean(x**2, axis=-1)).round(3)}")

print(f"\nAfter LayerNorm:")
print(f"  Mean: {ln_out.mean(axis=-1).round(6)}  (forced to ~0)")
print(f"  Std:  {ln_out.std(axis=-1).round(3)}  (forced to ~1)")

print(f"\nAfter RMSNorm:")
print(f"  Mean: {rms_out.mean(axis=-1).round(3)}  (NOT forced to 0)")
print(f"  RMS:  {np.sqrt(np.mean(rms_out**2, axis=-1)).round(3)}  (forced to ~1)")

print(f"\nKey difference: RMSNorm preserves the mean direction of the vector.")
print(f"It only controls the magnitude (RMS), not the center (mean).")
print(f"Parameters: LayerNorm has gamma AND beta ({2*dim}), RMSNorm has only gamma ({dim}).")

In [ ]:
# Visualize: what normalization does to activations through layers
np.random.seed(42)
n_layers = 10
dim = 64
x = np.random.randn(1, dim)

# Simulate passing through layers WITHOUT normalization
rms_no_norm = [np.sqrt(np.mean(x ** 2))]
rms_with_rms = [np.sqrt(np.mean(x ** 2))]

x_no_norm = x.copy()
x_rms = x.copy()
gamma = np.ones(dim)

for _ in range(n_layers):
    W = np.random.randn(dim, dim) * 0.5  # some random transformation
    
    x_no_norm = np.maximum(0, x_no_norm @ W)  # Linear + ReLU, no norm
    rms_no_norm.append(np.sqrt(np.mean(x_no_norm ** 2)))
    
    x_rms = rms_norm(np.maximum(0, x_rms @ W), gamma)  # Linear + ReLU + RMSNorm
    rms_with_rms.append(np.sqrt(np.mean(x_rms ** 2)))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(rms_no_norm, 'o-', color='#FF5722', linewidth=2, label='No normalization')
ax.plot(rms_with_rms, 's-', color='#2196F3', linewidth=2, label='With RMSNorm')
ax.set_xlabel('Layer', fontsize=12)
ax.set_ylabel('RMS of activations', fontsize=12)
ax.set_title('Activation Magnitude Through Layers', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Without normalization: activations shrink or explode through layers.")
print("With RMSNorm: activations stay controlled at magnitude ~1.")
print("This is essential for training deep networks (6+ layers) like our transformer.")

---
## Part 3: SwiGLU (Swish-Gated Linear Unit)

### The Feed-Forward Network (FFN)
In a transformer, after attention, each token passes through an FFN:
- **Old (GPT-2)**: `FFN(x) = GELU(x·W₁ + b₁)·W₂ + b₂`
- **Modern (Llama)**: `FFN(x) = (Swish(x·W_gate) ⊙ (x·W_up))·W_down`

### What's SwiGLU?
SwiGLU combines two ideas:
1. **Swish** activation: `swish(x) = x · sigmoid(x)` — smooth, non-monotonic
2. **Gating**: multiply element-wise with another projection

```
SwiGLU(x) = Swish(x · W_gate) ⊙ (x · W_up)
```

The **gate** controls how much information flows through, like a learned filter.
This gives 1-2% improvement over GELU at the same compute cost.

In [ ]:
def gelu(x):
    """Gaussian Error Linear Unit (GPT-2 era)."""
    return 0.5 * x * (1.0 + np.tanh(np.sqrt(2.0 / np.pi) * (x + 0.044715 * x ** 3)))

def swish(x):
    """Swish activation: x * sigmoid(x). Smooth, non-monotonic."""
    return x * (1.0 / (1.0 + np.exp(-x)))

def relu(x):
    return np.maximum(0, x)

# Compare activations
x = np.linspace(-5, 5, 200)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, fn, name, color in [
    (axes[0], relu, 'ReLU (basic)', '#9E9E9E'),
    (axes[1], gelu, 'GELU (GPT-2)', '#FF9800'),
    (axes[2], swish, 'Swish (Llama/Gemma)', '#2196F3'),
]:
    ax.plot(x, fn(x), color=color, linewidth=2.5)
    ax.axhline(y=0, color='gray', linewidth=0.5)
    ax.axvline(x=0, color='gray', linewidth=0.5)
    ax.set_title(name, fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.set_ylim(-1, 5)

plt.suptitle('Activation Functions: Evolution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Key difference: Swish allows small negative values through.")
print("This gives richer gradients and slightly better learning.")

In [ ]:
def old_ffn(x, W1, b1, W2, b2):
    """GPT-2 style FFN: GELU(x·W1+b1)·W2+b2
    Two weight matrices, two biases.
    """
    return gelu(x @ W1 + b1) @ W2 + b2


def swiglu_ffn(x, W_gate, W_up, W_down):
    """Modern SwiGLU FFN: (Swish(x·W_gate) ⊙ (x·W_up))·W_down
    Three weight matrices, no biases.
    
    The gate projection learns WHAT to filter.
    The up projection provides the CONTENT.
    The down projection reduces back to model dimension.
    """
    gate = swish(x @ W_gate)  # what to let through
    up = x @ W_up             # the content
    hidden = gate * up         # element-wise gating
    return hidden @ W_down     # project back to model dimension


# Parameter comparison
d_model = 384  # our Phase 3 model
d_ff_old = 4 * d_model      # GPT-2: 4x expansion
d_ff_new = int(2.67 * d_model)  # SwiGLU: ~2.67x (3 matrices instead of 2, so reduce to match params)

old_params = d_model * d_ff_old + d_ff_old + d_ff_old * d_model + d_model
new_params = 3 * d_model * d_ff_new  # 3 matrices, no bias

print("FFN Parameter Comparison:")
print(f"  GPT-2 (GELU):   d_ff={d_ff_old}, params={old_params:,}")
print(f"  Llama (SwiGLU):  d_ff={d_ff_new}, params={new_params:,}")
print(f"  Ratio: {new_params/old_params:.2f}x")
print(f"\nSwiGLU has ~similar parameters but better performance.")
print(f"The gating mechanism acts as a learned information filter.")

---
## Part 4: Grouped Query Attention (GQA)

### The Problem with Standard Multi-Head Attention (MHA)
In MHA, each head has its OWN Q, K, and V projections. For inference:
- The K and V tensors from all previous tokens must be cached (KV-cache)
- More heads = larger cache = more memory
- This becomes the bottleneck for long sequences

### GQA: Share K/V Heads
Instead of each query head having its own K/V, multiple query heads **share** K/V heads:

```
MHA:  Q heads: [0,1,2,3,4,5]  K/V heads: [0,1,2,3,4,5]  (1:1 ratio)
GQA:  Q heads: [0,1,2,3,4,5]  K/V heads: [0, 1]          (3:1 ratio)
MQA:  Q heads: [0,1,2,3,4,5]  K/V head:  [0]              (6:1 ratio)
```

Our Phase 3 model uses GQA with 6 Q heads and 2 KV heads.
This gives:
- **3x smaller KV-cache** than MHA
- **Quality nearly identical** to MHA
- **Faster inference** due to smaller memory reads

In [ ]:
def grouped_query_attention(X, W_Q, W_K, W_V, W_O, n_q_heads, n_kv_heads, mask=None):
    """Grouped Query Attention — the modern attention variant.
    
    Args:
        X: (seq_len, d_model)
        W_Q: (d_model, d_model)          — full Q projection
        W_K: (d_model, d_kv)             — smaller K projection
        W_V: (d_model, d_kv)             — smaller V projection
        W_O: (d_model, d_model)          — output projection
        n_q_heads: number of query heads
        n_kv_heads: number of key/value heads (< n_q_heads)
    """
    seq_len, d_model = X.shape
    d_head = d_model // n_q_heads
    n_groups = n_q_heads // n_kv_heads  # queries per KV head
    
    # Project
    Q = X @ W_Q  # (seq_len, d_model)
    K = X @ W_K  # (seq_len, d_kv)  where d_kv = n_kv_heads * d_head
    V = X @ W_V  # (seq_len, d_kv)
    
    # Reshape Q: (seq_len, n_q_heads, d_head) -> (n_q_heads, seq_len, d_head)
    Q = Q.reshape(seq_len, n_q_heads, d_head).transpose(1, 0, 2)
    
    # Reshape K, V: (seq_len, n_kv_heads, d_head) -> (n_kv_heads, seq_len, d_head)
    K = K.reshape(seq_len, n_kv_heads, d_head).transpose(1, 0, 2)
    V = V.reshape(seq_len, n_kv_heads, d_head).transpose(1, 0, 2)
    
    # Expand K, V to match Q heads by repeating
    # Each KV head is shared by n_groups query heads
    K_expanded = np.repeat(K, n_groups, axis=0)  # (n_q_heads, seq_len, d_head)
    V_expanded = np.repeat(V, n_groups, axis=0)
    
    # Standard attention per head
    scores = np.matmul(Q, K_expanded.transpose(0, 2, 1)) / np.sqrt(d_head)
    # scores: (n_q_heads, seq_len, seq_len)
    
    if mask is not None:
        scores = np.where(mask == 0, -1e9, scores)
    
    # Softmax
    exp_s = np.exp(scores - np.max(scores, axis=-1, keepdims=True))
    weights = exp_s / np.sum(exp_s, axis=-1, keepdims=True)
    
    # Weighted sum of values
    attn_out = np.matmul(weights, V_expanded)  # (n_q_heads, seq_len, d_head)
    
    # Concatenate heads: (seq_len, d_model)
    attn_out = attn_out.transpose(1, 0, 2).reshape(seq_len, d_model)
    
    # Output projection
    output = attn_out @ W_O
    
    return output, weights


# Demo: our Phase 3 architecture
d_model = 12  # small for demo (real: 384)
n_q_heads = 6
n_kv_heads = 2
d_head = d_model // n_q_heads  # 2
d_kv = n_kv_heads * d_head      # 4
seq_len = 4

np.random.seed(42)
X = np.random.randn(seq_len, d_model)
W_Q = np.random.randn(d_model, d_model) * 0.1
W_K = np.random.randn(d_model, d_kv) * 0.1
W_V = np.random.randn(d_model, d_kv) * 0.1
W_O = np.random.randn(d_model, d_model) * 0.1

mask = np.tril(np.ones((seq_len, seq_len)))
output, weights = grouped_query_attention(X, W_Q, W_K, W_V, W_O, n_q_heads, n_kv_heads, mask)

print(f"GQA Configuration:")
print(f"  Q heads:  {n_q_heads}")
print(f"  KV heads: {n_kv_heads}")
print(f"  Groups:   {n_q_heads // n_kv_heads} (Q heads per KV head)")
print(f"  d_head:   {d_head}")
print(f"\nOutput shape: {output.shape}")
print(f"Weights shape: {weights.shape} ({n_q_heads} heads, {seq_len}x{seq_len})")

In [ ]:
# Memory savings comparison
d_model = 384  # our Phase 3 model
n_q_heads = 6
d_head = d_model // n_q_heads  # 64
seq_len = 256  # context length

configs = [
    ('MHA (6 KV heads)', 6),
    ('GQA (2 KV heads)', 2),
    ('MQA (1 KV head)', 1),
]

print("KV-Cache Memory Comparison (per token, float32):")
print("=" * 55)
print(f"{'Config':<25} {'KV heads':>10} {'KV Cache':>15}")
print("-" * 55)

for name, n_kv in configs:
    # KV cache per token: 2 (K+V) * n_kv_heads * d_head * 4 bytes
    cache_per_token = 2 * n_kv * d_head * 4  # bytes
    total_cache = cache_per_token * seq_len
    print(f"{name:<25} {n_kv:>10} {total_cache/1024:>12.1f} KB")

print("-" * 55)
print(f"\nGQA (our choice) saves {6/2:.0f}x memory vs MHA with minimal quality loss.")
print(f"At 256 tokens: MHA={2*6*64*256*4/1024:.0f}KB vs GQA={2*2*64*256*4/1024:.0f}KB")

In [ ]:
# Visualize GQA head sharing
fig, ax = plt.subplots(figsize=(10, 3))

# Draw Q heads
q_colors = ['#1565C0', '#1976D2', '#2196F3', '#42A5F5', '#64B5F6', '#90CAF9']
for i in range(6):
    ax.add_patch(plt.Rectangle((i * 1.5, 2), 1.2, 0.8, color=q_colors[i], alpha=0.8))
    ax.text(i * 1.5 + 0.6, 2.4, f'Q{i}', ha='center', va='center', fontweight='bold', fontsize=10)

# Draw KV heads (only 2)
kv_colors = ['#E65100', '#FF6D00']
kv_positions = [1.5, 6.0]  # centered under their Q groups
for i, pos in enumerate(kv_positions):
    ax.add_patch(plt.Rectangle((pos, 0), 1.2, 0.8, color=kv_colors[i], alpha=0.8))
    ax.text(pos + 0.6, 0.4, f'KV{i}', ha='center', va='center', 
            fontweight='bold', fontsize=10, color='white')

# Draw arrows from Q to KV
for i in range(3):
    ax.annotate('', xy=(2.1, 0.8), xytext=(i * 1.5 + 0.6, 2),
               arrowprops=dict(arrowstyle='->', color=kv_colors[0], lw=1.5))
for i in range(3, 6):
    ax.annotate('', xy=(6.6, 0.8), xytext=(i * 1.5 + 0.6, 2),
               arrowprops=dict(arrowstyle='->', color=kv_colors[1], lw=1.5))

ax.set_xlim(-0.5, 10)
ax.set_ylim(-0.5, 3.5)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Grouped Query Attention: 6 Q heads share 2 KV heads (3:1 ratio)',
             fontsize=13, fontweight='bold')
ax.text(4.5, -0.3, 'Q0,Q1,Q2 share KV0    |    Q3,Q4,Q5 share KV1', 
        ha='center', fontsize=10, style='italic')
plt.tight_layout()
plt.show()

---
## Part 5: Putting It All Together — The Modern Transformer Block

Combining all four components, a modern transformer block looks like:

```
x → RMSNorm → GQA (with RoPE on Q,K) → + residual
  → RMSNorm → SwiGLU FFN               → + residual
```

Compare with GPT-2:
```
x → LayerNorm → MHA (with learned pos emb) → + residual
  → LayerNorm → GELU FFN                    → + residual
```

Same structure, better components. This is what we'll implement in Phase 3.

In [ ]:
print("=" * 65)
print("Summary: Modern vs Old Transformer Components")
print("=" * 65)
print(f"{'Component':<20} {'GPT-2 (2019)':<22} {'Llama/Our Model (2025)'}")
print("-" * 65)
print(f"{'Position':<20} {'Learned embeddings':<22} {'RoPE (rotation)'}")
print(f"{'Normalization':<20} {'LayerNorm':<22} {'RMSNorm'}")
print(f"{'Attention':<20} {'MHA':<22} {'GQA (shared KV)'}")
print(f"{'FFN activation':<20} {'GELU':<22} {'SwiGLU (gated)'}")
print(f"{'Biases':<20} {'Yes':<22} {'No (removed)'}")
print(f"{'LR schedule':<20} {'Cosine decay':<22} {'WSD (warmup-stable-decay)'}")
print("=" * 65)
print("\nAll components implemented from scratch in numpy.")
print("Next: Phase 2 (Tokenizer) and Phase 3 (full PyTorch implementation).")

## Summary & Key Takeaways

1. **RoPE** encodes position by rotation, making attention depend on relative position naturally
2. **RMSNorm** is simpler and faster than LayerNorm — just normalize by RMS, no mean subtraction
3. **SwiGLU** uses gating to filter information — 1-2% better than GELU at same cost
4. **GQA** shares K/V heads across Q heads — 3x smaller KV-cache, nearly same quality
5. These four changes are what separate a 2019 GPT-2 from a 2025 Llama/Gemma model
6. The overall transformer structure (attention + FFN + residuals) is unchanged — only the components improved

### What's Next
- **Phase 2**: Build the BPE tokenizer (can do in parallel with Phase 3)
- **Phase 3**: Implement all these components in PyTorch as proper `nn.Module` classes
- **Phase 4**: Train the model for real